# Wan 2.2 TI2V-5B — moving shots (AI Heritage Short)
Animates each keyframe with its motion prompt. All logic lives in `wan2_ti2v_runner.py` (uploaded in the
`heritage-short-inputs` dataset). **Settings:** Accelerator = GPU (T4 x2 / P100), Internet = On.
The run **fails loudly** if the model can't load. It never writes placeholder videos.

In [ ]:
# 1. Dependencies. Wan2.2 TI2V needs a recent diffusers (the model card says: main branch).
!pip install -q -U git+https://github.com/huggingface/diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg ftfy

In [ ]:
# 2. GPU check: stop here if no GPU.
import torch
assert torch.cuda.is_available(), "No GPU: set Accelerator to GPU T4 x2 or P100 and restart the session."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/2**30,1), "GB")

In [ ]:
# 3. Locate the uploaded dataset (wan_batch.json, keyframes/, wan2_ti2v_runner.py).
import sys
from pathlib import Path
DATA = next((p for p in Path("/kaggle/input").glob("*") if (p/"wan_batch.json").is_file()), None)
assert DATA, "Dataset not attached: add the `heritage-short-inputs` dataset to this notebook."
sys.path.insert(0, str(DATA))
print("Using", DATA, "->", sorted(x.name for x in DATA.iterdir()))

In [ ]:
# 4. SMOKE TEST: generate ONE shot first. Check quality, time and memory before the full batch.
#    If you hit CUDA out-of-memory, lower MAX_AREA / MAX_FRAMES (they are in the report).
import wan2_ti2v_runner as w
MAX_FRAMES = 121          # 4k+1; 121 = 5 s @ 24 fps
MAX_AREA   = 480 * 832    # pixels per frame
STEPS      = 30
OUT = "/kaggle/working/outputs/shots"
rep = w.run_wan_batch(DATA/"wan_batch.json", DATA/"keyframes", OUT, only=["shot_01"],
                      overrides=dict(max_frames=MAX_FRAMES, max_area=MAX_AREA, num_inference_steps=STEPS))
print(rep["shots"])

In [ ]:
# 5. Preview the first clip.
from IPython.display import Video
Video(OUT + "/shot_01.mp4", embed=True, width=480)

In [ ]:
# 6. FULL BATCH (skips nothing, regenerates shot_01 too). Kaggle sessions are limited (~9-12 h);
#    run_report.json is saved after every shot so partial progress survives a timeout.
rep = w.run_wan_batch(DATA/"wan_batch.json", DATA/"keyframes", OUT,
                      overrides=dict(max_frames=MAX_FRAMES, max_area=MAX_AREA, num_inference_steps=STEPS))

In [ ]:
# 7. Package clips + measured report for download. Then run on your PC:
#    python kaggle/bundle.py unpack outputs_shots.zip
import shutil
shutil.make_archive("/kaggle/working/outputs_shots", "zip", OUT)
print("Download /kaggle/working/outputs_shots.zip from the Output tab.")